In [1]:
import Pkg

In [2]:
Pkg.activate(".")
Pkg.add("AppleAccelerate")
Pkg.add("ThreadPinning")
Pkg.add("KrylovKit")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.add("QuantumControl")
Pkg.instantiate()

  Activating project at `~/Documents/Research/2025-05_KrylovKitBenchmark`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_

In [3]:
using Random
using KrylovKit: expintegrator, Arnoldi
using BenchmarkTools
using ProfileCanvas
using LinearAlgebra: norm
using QuantumControl: @threadsif

In [4]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	ophelia
CPU(s): 	1 x Apple M4 Max
CPU target: 	apple-m1
Cores: 		16 (16 CPU-threads)
Core kinds: 	4 "efficiency cores", 12 "performance cores".
NUMA domains: 	1 (16 cores each)

Unsupported OS: Won't be able to highlight Julia threads.

Julia threads: 	8

CPU socket 1
  0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15


# = Julia thread, # = >1 Julia thread, # = Efficiency core


In [5]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "8"

In [6]:
using AppleAccelerate

In [7]:
N = 100

100

In [8]:
"""Random complex matrix of dimension `N` with a spectral radius of approximately `ρ`."""
function random_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    H = ρ * (X + Y * 1im) / √2
    return H
end

function random_hermitian_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    Z = (X + Y * 1im) / √2
    H = ρ * (Z + Z') / (2 * √2)
    return H
end


"""Random normalized complex vector of dimension `N`"""
function random_state_vector(N=N; rng=Random.GLOBAL_RNG)
    Ψ = rand(rng, N) .* exp.((2π * im) .* rand(rng, N))
    Ψ ./= norm(Ψ)
    return Ψ
end

random_state_vector

In [9]:
struct Trajectory
    initial_state::Vector{ComplexF64}
    H::Matrix{ComplexF64}
    dt::Vector{Float64}
end

function Trajectory(;initial_state, H, nt)
    dt = rand(nt)
    N = length(initial_state)
    @assert size(H) == (N, N)
    Trajectory(initial_state, H, dt)
end

Trajectory

In [10]:
function propagate_trajs(trajs::Vector{Trajectory}, use_threads::Bool)
    # even if traj.H is Hermitian, we still use Arnoldi.
    # This propagation method is intended for non-Hermitian generators,
    # using a general matrix screws up the benchmark because the norm
    # of Ψ explodes.
    alg = Arnoldi()
    numops_total = Threads.Atomic{Int64}(0)
    @threadsif use_threads for traj in trajs
        Ψ = traj.initial_state
        numops = 0
        for dt in traj.dt
            Ψ, info = expintegrator(traj.H, -1im * dt, (Ψ, ), alg)
            numops += info.numops
        end
        Threads.atomic_add!(numops_total, numops)
    end
    return numops_total[]
end

function propagate_trajs(trajs::Vector{Trajectory}; use_threads=true)
    return propagate_trajs(trajs, use_threads)
end


propagate_trajs (generic function with 2 methods)

In [11]:
trajs = [
    Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=100)
    for _ in 1:Threads.nthreads()
]
propagate_trajs(trajs)

24800

In [12]:
@code_warntype propagate_trajs(trajs, false)

MethodInstance for propagate_trajs(::Vector{Trajectory}, ::Bool)
  from propagate_trajs(trajs::Vector{Trajectory}, use_threads::Bool) @ Main In[10]:1
Arguments
  #self#::Core.Const(Main.propagate_trajs)
  trajs::Vector{Trajectory}
  use_threads::Bool
Locals
  @_4::Union{Nothing, Tuple{Trajectory, Int64}}
  threadsfor_fun::var"#6#threadsfor_fun#8"{var"#6#threadsfor_fun#7#9"{Base.Threads.Atomic{Int64}, Arnoldi{KrylovKit.ModifiedGramSchmidt2, Float64}, Vector{Trajectory}}}
  numops_total::Base.Threads.Atomic{Int64}
  alg::Arnoldi{KrylovKit.ModifiedGramSchmidt2, Float64}
  threadsfor_fun#7::var"#6#threadsfor_fun#7#9"{Base.Threads.Atomic{Int64}, Arnoldi{KrylovKit.ModifiedGramSchmidt2, Float64}, Vector{Trajectory}}
  range::Vector{Trajectory}
  @_10::Union{Nothing, Tuple{Float64, Int64}}
  traj::Trajectory
  numops::Int64
  Ψ::Vector{ComplexF64}
  @_14::Int64
  dt::Float64
  info::KrylovKit.ConvergenceInfo{Float64, Float64}
Body::Int64
1 ──        Core.NewvarNode(:(@_4))
│           Core.New

In [13]:
bm_sequential = @benchmark propagate_trajs($trajs; use_threads=false)

BenchmarkTools.Trial: 24 samples with 1 evaluation.
 Range (min … max):  210.549 ms … 236.294 ms  ┊ GC (min … max): 2.16% … 2.15%
 Time  (median):     211.974 ms               ┊ GC (median):    2.18%
 Time  (mean ± σ):   214.624 ms ±   6.403 ms  ┊ GC (mean ± σ):  2.20% ± 0.46%

  ▂█▂▂                                                           
  █████▅▅▁▁▁▁▁▅▁▁▁▁▁▅▁▁▅▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅ ▁
  211 ms           Histogram: frequency by time          236 ms <

 Memory estimate: 157.69 MiB, allocs estimate: 128805.

In [14]:
bm_parallel = @benchmark propagate_trajs($trajs)

BenchmarkTools.Trial: 149 samples with 1 evaluation.
 Range (min … max):  31.052 ms …  36.563 ms  ┊ GC (min … max):  5.24% … 16.91%
 Time  (median):     33.486 ms               ┊ GC (median):    11.99%
 Time  (mean ± σ):   33.590 ms ± 749.811 μs  ┊ GC (mean ± σ):  12.09% ±  1.47%

                      ▁  ▁▇▆▃▆▃▃█  ▃                            
  ▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▆▆▇████████████▆██▇█▃▄▄▁▁▃▆▁▁▁▄▁▁▃▃▁▃▁▃▁▁▁▃ ▃
  31.1 ms         Histogram: frequency by time         36.3 ms <

 Memory estimate: 157.69 MiB, allocs estimate: 128843.

In [15]:
mean(bm_sequential.times) / mean(bm_parallel.times)

6.3894682933302445

In [16]:
Base.GC.enable(false)
bm_parallel =  @benchmark propagate_trajs($trajs)

BenchmarkTools.Trial: 133 samples with 1 evaluation.
 Range (min … max):  32.070 ms … 75.348 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     36.348 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   37.764 ms ±  5.169 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

     ▂ ▂█▆▇▃ ▃     ▂                                           
  ▃▄▅█▇████████▆▅█▅█▄▄▁▇▄▄▃▁▁▁▃▁▁▁▁▁▃▁▁▁▃▁▁▃▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▃ ▃
  32.1 ms         Histogram: frequency by time        58.6 ms <

 Memory estimate: 157.69 MiB, allocs estimate: 128843.

In [17]:
Base.GC.enable(true);

In [18]:
mean(bm_sequential.times) / mean(bm_parallel.times)

5.683380873934909

In [19]:
@profview propagate_trajs(trajs)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("3" => ProfileCanvas.ProfileFrame("root", "", "", 0, 24, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#1", "threadingconstructs.jl", "./threadingconstructs.jl", 154, 16, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#6#threadsfor_fun", "threadingconstructs.jl", "./threadingconstructs.jl", 220, 16, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#6#threadsfor_fun#7", "threadingconstructs.jl", "./threadingconstructs.jl", 253, 16, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("macro expansion", "In[10]", "./In[10]", 12, 16, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/Users/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 106, 16, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/Users/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 189, 9, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("exp!", "dense.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/dense.jl", 717, 5, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("gesv!", "lapack.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/lapack.jl", 1028, 5, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("exp!", "dense.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/dense.jl", 711, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("*", "matmul.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/matmul.jl", 130, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("similar", "array.jl", "./array.jl", 372, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("Array", "boot.jl", "./boot.jl", 592, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("Array", "boot.jl", "./boot.jl", 582, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("new_as_memoryref", "boot.jl", "./boot.jl", 535, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("GenericMemory", "boot.jl", "./boot.jl", 516, 1, missing, 0x02, missing, ProfileCanvas.ProfileFrame[])])])])])])]), ProfileCanvas.ProfileFrame("exp!", "dense.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/dense.jl", 699, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("*", "matmul.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/matmul.jl", 130, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("mul!", "matmul.jl", "/Users/julia/.julia/scr